Họ và tên: Nguyễn Trọng Phúc, MSSV: 2510194

## Bài 1: Titanic Dataset

Sử dụng lại bộ dữ liệu **Titanic** trong bài tập về nhà trước.

### Yêu cầu

1. Sử dụng **Logistic Regression** để dự đoán hành khách sống sót hay không.
2. Sử dụng cùng cách chia dữ liệu train/test như bài tập trước nếu có thể.
3. Đánh giá kết quả của mô hình Logistic Regression.
4. So sánh kết quả của **Logistic Regression** với kết quả của **Linear Regression** trong bài tập trước.

### Nội dung so sánh

Có thể so sánh dựa trên các chỉ số:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

Đưa ra nhận xét về mô hình phù hợp hơn đối với bài toán phân loại hành khách sống sót.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [2]:
#Load dữ liệu và loại bỏ cột thừa
df = sns.load_dataset('titanic') 
leaky = ['class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
df = df.drop(columns = leaky)

#Chia tập
X = df.drop(columns = ['survived'])
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

#Xây pipeline và preprocessing
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so = Pipeline([
    ("imputer", SimpleImputer(strategy = 'median')),
    ("scaler", RobustScaler()),
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore', sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", pipe_so, num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)
X_train_t = preprocess.transform(X_train)
X_test_t = preprocess.transform(X_test)

# --------- LINEAR REGRESSION -----------
linear_model = LinearRegression()
linear_model.fit(X_train_t, y_train)

y_pred_linear = linear_model.predict(X_test_t)
y_pred_linear_convert = np.where(y_pred_linear >= 0.5, 1, 0)

# --------- LOGISTIC REGRESSION ----------
logistic_model = LogisticRegression()
logistic_model.fit(X_train_t, y_train)

y_pred_logistic = logistic_model.predict(X_test_t)

# --------- ĐÁNH GIÁ VÀ SO SÁNH ----------

#Confusion Matrix, Accuracy, Precision, Recall, F1-score của Logistic regression
print('Confusion Matrix, Accuracy, Precision, Recall, F1-score của Logistic regression')

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_logistic))
print('Accuracy:', accuracy_score(y_test, y_pred_logistic))
print('Precision:', precision_score(y_test, y_pred_logistic))
print('Recall:', recall_score(y_test, y_pred_logistic))
print('F1-Score:', f1_score(y_test, y_pred_logistic))

#Confusion Matrix, Accuracy, Precision, Recall, F1-score của Linear regression
print('Confusion Matrix, Accuracy, Precision, Recall, F1-score của Linear regression')

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_linear_convert))
print('Accuracy:', accuracy_score(y_test, y_pred_linear_convert))
print('Precision:', precision_score(y_test, y_pred_linear_convert))
print('Recall:', recall_score(y_test, y_pred_linear_convert))
print('F1-Score:', f1_score(y_test, y_pred_linear_convert))

Confusion Matrix, Accuracy, Precision, Recall, F1-score của Logistic regression
Confusion Matrix:
[[98 12]
 [23 46]]
Accuracy: 0.8044692737430168
Precision: 0.7931034482758621
Recall: 0.6666666666666666
F1-Score: 0.7244094488188977
Confusion Matrix, Accuracy, Precision, Recall, F1-score của Linear regression
Confusion Matrix:
[[97 13]
 [21 48]]
Accuracy: 0.8100558659217877
Precision: 0.7868852459016393
Recall: 0.6956521739130435
F1-Score: 0.7384615384615385


## Nhận xét:
Trong trường hợp này, có thể thấy Linear Regression hay Logistic Regression cho điểm số không có cách biệt lớn, nhưng về mặt bản chất thì Logistic Regression mới là mô hình phù hợp và nên được lựa chọn trong bài toán phân loại nhị phân, cụ thể là phân loại hành khách sống sót, vì các lí do sau:
- Logistic regression trực tiếp học một ranh giới quyết định tối ưu để tách biệt các lớp dữ liệu. Ngược lại, linear regression chỉ đơn thuần cố gắng "khớp" một đường thẳng sao cho khoảng cách bình phương tới các điểm 0 và 1 là nhỏ nhất. Việc tối ưu hóa một đường thẳng hồi quy hoàn toàn đi lệch với mục tiêu phân loại của bài toán.
- Linear regression sử dụng MSE làm loss function, MSE sẽ phạt cả những dự đoán "tự tin thái quá", ví dụ như mô hình dự đoán ra 1.8 cho nhãn 1, sai số bị tính là 0.8^2, một sai số không lớn, điều này vô lý vì xác suất chỉ nằm trong [0,1]. Trong khi đó, Logistic Regression sử dụng BCE, hàm này sử dụng trên xác suất, nó sẽ không phạt việc dự đoán đúng lớp, mà chỉ phạt rất nặng khi mô hình dự đoán cực kỳ tự tin nhưng lại sai bét (ví dụ dự đoán 0.99 cho nhãn 0).

## Bài 2: Dry Bean Dataset

Xây dựng mô hình phân loại các loại hạt trong bộ dữ liệu **Dry Bean Dataset** bằng hai thuật toán:

1. **Logistic Regression**
2. **K-Nearest Neighbors – KNN**

In [6]:
from sklearn.neighbors import KNeighborsClassifier

#Tải dữ liệu 
train_df = pd.read_csv('Dry_Bean_Dataset/dry_bean_train.csv')
test_df = pd.read_csv('Dry_Bean_Dataset/dry_bean_test.csv')

#Tách đặc trưng X và mục tiêu y:
X_train_new = train_df.drop(columns = ['class'])
y_train_new = train_df['class']
X_test_new = test_df.drop(columns = ['class'])
y_test_new = test_df['class']

#Xây pipeline và training model
#--------- LOGISTIC REGRESSION -----------
pipe_log = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression())
])

pipe_log.fit(X_train_new, y_train_new)
y_pred_logistic_new = pipe_log.predict(X_test_new)


#--------- KNN ---------
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

pipe_knn.fit(X_train_new, y_train_new)
y_pred_knn = pipe_knn.predict(X_test_new)


#Đánh giá, so sánh
print('Kết quả của Logistic Regression:')

print('Confusion Matrix:')
print(confusion_matrix(y_test_new, y_pred_logistic_new))
print('Accuracy:', accuracy_score(y_test_new, y_pred_logistic_new))
print('Precision:', precision_score(y_test_new, y_pred_logistic_new, average='macro'))
print('Recall:', recall_score(y_test_new, y_pred_logistic_new, average='macro'))
print('F1-Score:', f1_score(y_test_new, y_pred_logistic_new, average='macro'))

print('\nKết quả của K-Nearest Neighbours:')

print('Confusion Matrix:')
print(confusion_matrix(y_test_new, y_pred_knn))
print('Accuracy:', accuracy_score(y_test_new, y_pred_knn))
print('Precision:', precision_score(y_test_new, y_pred_knn, average='macro'))
print('Recall:', recall_score(y_test_new, y_pred_knn, average='macro'))
print('F1-Score:', f1_score(y_test_new, y_pred_knn, average='macro'))

Kết quả của Logistic Regression:
Confusion Matrix:
[[236   0  18   0   0   4   7]
 [  0 104   0   0   0   0   0]
 [  8   0 307   0   5   2   4]
 [  0   0   0 643   0  13  53]
 [  1   0  11   5 351   0   4]
 [  9   0   0   5   0 383   9]
 [  1   0   1  41   8  10 466]]
Accuracy: 0.9191583610188261
Precision: 0.9307249956317589
Recall: 0.9300490838251013
F1-Score: 0.9302365421734592

Kết quả của K-Nearest Neighbours:
Confusion Matrix:
[[232   0  21   0   1   3   8]
 [  0 104   0   0   0   0   0]
 [  8   0 309   0   5   2   2]
 [  0   0   0 647   0  10  52]
 [  0   0  13   4 347   0   8]
 [  6   0   1   7   0 380  12]
 [  3   0   0  49   8   6 461]]
Accuracy: 0.9154669619785899
Precision: 0.9289934228682724
Recall: 0.9256280640932312
F1-Score: 0.9270270638216341


Nhìn vào kết quả, ta rút ra được một số điều sau:
- Ở confusion matrix ở 2 mô hình, ta thấy cả 2 cách đều bị nhầm lẫn giữa lớp 3 sang lớp 6 và ngược lại với số lượng mẫu đáng kể. Có vẻ như dữ liệu của 2 loại hạt 'CALI' và 'DERMASON' có các đặc trưng như 'area' hay 'perimeter' chồng lấn nhau trong không gian nên 2 mô hình khó phân tách hoàn toàn.
- Trong khi logistic regression học và điều chỉnh ranh giới quyết định trong không gian đặc trưng thì KNN lại chỉ dựa vào khoảng cách cục bộ, nhưng cả 2 cách tiếp cận đều cho ra các pattern giống nhau trong kết quả, điều này xảy ra là vì dữ liệu hiện tại có các thông số đo đạc chưa đủ khả năng để tách biệt hoàn toàn các loại hạt. 